In [86]:
import pandas as pd
import numpy as np
from statsmodels.stats.inter_rater import fleiss_kappa
from statsmodels.stats.inter_rater import aggregate_raters
from sklearn.metrics import cohen_kappa_score

In [ ]:
df = pd.read_csv("annotations_combined.csv").drop(["img_id", "group_id"] , axis=1)
df = df.drop(351 , axis=0).reset_index(drop=True)     # drop a NaN row

all_pen = df[["pen_1", "pen_2", "pen_3", "pen_4", "pen_5"]]             # NaN from the fifth pen annotator is included
all_hair = df[["hair_1", "hair_2", "hair_3", "hair_4", "hair_5"]]       # NaN from the fifth hair annotator is included
all_4_pen = df[["pen_1", "pen_2", "pen_3", "pen_4"]]                # The BEST for pen with no NaN values
all_4_hair = df[["hair_1", "hair_2", "hair_3", "hair_4"]]           # The BEST for hair with no NaN values


# The following code only considers the data with the fifth annotator included
df_5 = df[df["hair_5"].notna()].dropna()     # there is a missing value    # Should I add .reset_index(drop=True)?
five_pen = df_5[["pen_1", "pen_2", "pen_3", "pen_4", "pen_5"]]
five_hair = df_5[["hair_1", "hair_2", "hair_3", "hair_4", "hair_5"]]

In [ ]:
# Fleiss Kappa scores
# WARNING: Fleiss Kappa only works for nominal data, so it works only for pen marks and not for amount of hair

print(f"Fleiss Kappa for all pen annotators: \t {fleiss_kappa(aggregate_raters(all_pen)[0])}")
print(f"Fleiss Kappa for all 4 pen annotators: \t {fleiss_kappa(aggregate_raters(all_4_pen)[0])}")      # the Best one
print(f"Fleiss Kappa for 5 pen annotators: \t {fleiss_kappa(aggregate_raters(five_pen)[0])}")

Fleiss Kappa for all pen annotators: 	 0.7819796314938243
Fleiss Kappa for all 4 pen annotators: 	 0.8180090858187538
Fleiss Kappa for 5 pen annotators: 	 0.8730560215728854


In [89]:
# Cohen Cappa scores for all annotators
# Is it a good idea to include the fifth annotator despite the missing data?

average_pen = 0
cohen_scores_pen = np.zeros((5,5))
for i in range(1,5):
    for j in range(i+1,5):
        cohen_scores_pen[i-1,j-1] = cohen_kappa_score(all_4_pen[f"pen_{i}"], all_4_pen[f"pen_{j}"])
#        cohen_scores_pen[j-1,i-1] = cohen_scores_pen[i-1,j-1]           # just for symmetry
        average_pen += cohen_scores_pen[i-1,j-1]
for i in range(1, 5):           # Adding the fifth annotator separately due to missing values
    cohen_scores_pen[i-1,4] = cohen_kappa_score(five_pen[f"pen_{i}"], five_pen["pen_5"])
    average_pen += cohen_scores_pen[i-1,4]

print(f"Cohen Kappa scores for pen: \n {cohen_scores_pen.round(3)} \n")

# Use linear in this case, according to https://www.medcalc.org/en/manual/kappa.php
average_hair = 0
cohen_scores_hair = np.zeros((5,5))
for i in range(1,5):
    for j in range(i+1,5):
        cohen_scores_hair[i-1,j-1] = cohen_kappa_score(all_4_hair[f"hair_{i}"], all_4_hair[f"hair_{j}"], weights="linear")
#        cohen_scores_hair[j-1,i-1] = cohen_scores_hair[i-1,j-1]        # just for symmetry
        average_hair += cohen_scores_hair[i-1,j-1]
for i in range(1, 5):          # Adding the fifth annotator separately due to missing values
    cohen_scores_hair[i-1,4] = cohen_kappa_score(five_hair[f"hair_{i}"], five_hair["hair_5"])
    average_hair += cohen_scores_hair[i-1,4]

print(f"Cohen Kappa scores for hair: \n {cohen_scores_hair.round(3)} \n")


print(f"Average score for pen: {round(average_pen/10, 3)}")
print(f"Average score for hair: {round(average_hair/10, 3)}")

Cohen Kappa scores for pen: 
 [[0.    0.9   0.908 0.756 0.913]
 [0.    0.    0.875 0.733 0.904]
 [0.    0.    0.    0.749 0.89 ]
 [0.    0.    0.    0.    0.834]
 [0.    0.    0.    0.    0.   ]] 

Cohen Kappa scores for hair: 
 [[0.    0.765 0.741 0.714 0.614]
 [0.    0.    0.731 0.71  0.622]
 [0.    0.    0.    0.789 0.633]
 [0.    0.    0.    0.    0.631]
 [0.    0.    0.    0.    0.   ]] 

Average score for pen: 0.846
Average score for hair: 0.695


In [90]:
# Valentina's previous group method for Fleiss Kappa (Using fleiss kappa for hair is wrong though)

# From my observation, the fleiss kappa score of this code is identical to the fleiss kappa score using aggregate_raters method
# Since the code below is very clumsy, it can be ignored 


annotations = pd.read_csv("annotations_combined.csv").drop(["img_id"] , axis=1)
annotations = annotations.drop(351 , axis=0).reset_index(drop=True)     # drop a NaN row
annotations = annotations[annotations["group_id"] == "f"].drop("group_id", axis=1)     # Valentina's previous group
#annotations = annotations[["pen_1", "pen_2", "pen_3", "pen_4"]]          # all pen annotations without the fifth annotator
#annotations = df[["hair_1", "hair_2", "hair_3", "hair_4"]]     # all hair annotations without the fifth annotator


#cols_hair = ["hair_1", "hair_2", "hair_3", "hair_4", "hair_5"]
cols_hair = ["hair_1", "hair_2", "hair_3", "hair_4"]           # for 4 annotators
data_h = annotations[cols_hair]
categories_h = sorted(pd.unique(data_h.values.ravel()))
table = []
for _, row in data_h.iterrows():
    counts = [(row == c).sum() for c in categories_h]
    table.append(counts)
kappa_h = fleiss_kappa(table)

print("Fleiss' Kappa (Hair):", kappa_h)

# cols_pen = ['pen_1', 'pen_2','pen_3', 'pen_4', 'pen_5']
cols_pen = ['pen_1', 'pen_2','pen_3', 'pen_4']             # for 4 annotators
data_p = annotations[cols_pen]
categories_p = sorted(pd.unique(data_p.values.ravel()))
table = []
for _, row in data_p.iterrows():
    counts = [(row == c).sum() for c in categories_p]
    table.append(counts)
kappa_p = fleiss_kappa(table)

print("Fleiss' Kappa (Hair):", kappa_p)

Fleiss' Kappa (Hair): 0.4891536273115219
Fleiss' Kappa (Hair): 0.879128329297821
